# Intro to Xarray

## Outline

1. Introduction to Xarray
2. Anatomy of a DataArray

While Pandas is great for many purposes, extending its inherently 2D data representation to a multidimensional dataset, such as 4-dimensional (time, vertical level, longitude, latitude) gridded Numerical Weather Prediction (NWP) or climate model output, is not efficient. 

Gridded data is typically written in a non-ASCII (i.e., not human-readable) format. The two most widely-used formats are [GRIB](https://en.wikipedia.org/wiki/GRIB) and [NetCDF](https://www.unidata.ucar.edu/software/netcdf/). Another format, quickly gaining traction, is [Zarr](https://zarr.readthedocs.io). These binary formats take up less storage space than text.

  - [GRIB (GRIdded Binary)](https://en.wikipedia.org/wiki/GRIB): Designed primarily for meteorological and weather forecast data. It is compact and efficient for storing large gridded fields such as temperature, wind, and pressure, but its metadata conventions can be less intuitive and more specialized.

  - [NetCDF (Network Common Data Form)](https://www.unidata.ucar.edu/software/netcdf/): A general purpose scientific format widely used for weather observations, climate, atmospheric, ocean, and other Earth-system datasets. It stores multidimensional arrays with descriptive metadata and is highly portable, but large files can become inefficient for parallel or cloud-based access. The [New York State Mesonet (NYSM)](https://nysmesonet.org/) data are primarily in NetCDF format.

  - [Zarr](https://zarr.readthedocs.io): Designed for chunked, compressed multidimensional arrays, especially for parallel computing and cloud storage. Unlike traditional NetCDF files, a Zarr dataset is typically stored as many chunks, making it well suited for scalable access to very large datasets.

## ERA5

[ECMWF Reanalysis, 5th generation (ERA5)](https://en.wikipedia.org/wiki/ECMWF_re-analysis) is a global atmospheric reanalysis produced by [ECMWF](https://en.wikipedia.org/wiki/European_Centre_for_Medium-Range_Weather_Forecasts) that combines historical observations with short-range forecasts from the ECMWF Integrated Forecasting System (IFS). This produces a spatially and temporally consistent reconstruction of past weather conditions, including variables such as temperature, pressure, wind, and precipitation.

### ERA5 Data Availability

ERA5 data is available through the [Climate Data Store](https://cds.climate.copernicus.eu/datasets).

We will use an example set of data from the [ERA5 hourly data on pressure levels from 1940 to present](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-pressure-levels?tab=overview)

#### Download ERA5 Data

1. Go to [ERA5 hourly data on pressure levels from 1940 to present](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-pressure-levels?)
2. Login or register to accept the terms of use
3. Select the `Download` tab
4. Product type: `Reanalysis`
5. Variable: `Select all`
6. Select a single Year, Month, and Day
7. Select the times of interest
8. Select a few pressure levels
9. Select the Geographical area of interest, e.g., rough northeast bounds (North: 48, South: 37, West: -82, East: -67) 
10. Data format: `NetCDF4`
11. Download format: `Unarchived`
12. Submit form
13. Wait for the request to complete and add the data to this directory as an NetCDF4 (.nc) file.

## Imports

In [ ]:
import xarray as xr

## Xarray
The Xarray package is well suited for gridded data like NetCDF and Zarr. It builds and extends on the multi-dimensional data structure in NumPy. We will see that some of the same methods we've used for Pandas have analogues in Xarray.

Below is part of a section from the [Xarray Tutorial](https://github.com/xarray-contrib/xarray-tutorial/tree/master/scipy-tutorial) that was presented at the [SciPy 2020 conference](https://www.scipy2020.scipy.org/):

***
<span style="color:green">(Start of SciPy 2020 Xarray Tutorial snippet)</span>
***

Multi-dimensional (a.k.a. N-dimensional, ND) arrays (sometimes called “tensors”)
are an essential part of computational science. They are encountered in a wide
range of fields, including physics, astronomy, geoscience, bioinformatics,
engineering, finance, and deep learning. In Python, [NumPy](https://numpy.org/)
provides the fundamental data structure and API for working with raw ND arrays.
However, real-world datasets are usually more than just raw numbers; they have
*labels* which encode information about how the array values map to locations in
space, time, etc.

Here is an example of how we might structure a dataset for a weather forecast:

<img src="http://xarray.pydata.org/en/stable/_images/dataset-diagram.png" align="center" width="80%">

You'll notice multiple *data variables* (temperature, precipitation), *coordinate
variables* (latitude, longitude), and *dimensions* (x, y, t). We'll cover how these
fit into Xarray's data structures below.

Xarray doesn’t just keep track of labels on arrays – it uses them to provide a
powerful and concise interface, with methods that look a lot like Pandas. For example:

- Apply operations over dimensions by name: `x.sum('time')`.

- Select values by label (or logical location) instead of integer location:
  `x.loc['2014-01-01']` or `x.sel(time='2014-01-01')`.

- Mathematical operations (e.g., `x - y`) vectorize across multiple dimensions
  (array broadcasting) based on dimension names, not shape.

- Easily use the split-apply-combine paradigm with groupby:
  `x.groupby('time.dayofyear').mean()`.

- Database-like alignment based on coordinate labels that smoothly handles
  missing values: `x, y = xr.align(x, y, join='outer')`.

- Keep track of arbitrary metadata in the form of a Python dictionary:
  `x.attrs`.

The N-dimensional nature of xarray’s data structures makes it suitable for
dealing with multi-dimensional scientific data, and its use of dimension names
instead of axis labels (`dim='time'` instead of `axis=0` or `axis='columns'`) makes such arrays much
more manageable than the raw numpy ndarray: with xarray, you don’t need to keep
track of the order of an array’s dimensions or insert dummy dimensions of size 1
to align arrays (e.g., using np.newaxis).

The immediate payoff of using xarray is that you’ll write less code. The
long-term payoff is that you’ll understand what you were thinking when you come
back to look at it weeks or months later.

***
<span style="color:green">(End of SciPy 2020 Xarray Tutorial snippet)</span>
***



### Core XArray objects: the **Dataset** and the **DataArray**

As with Pandas, which has `Series` and `DataFrame` as its core data structures, Xarray has two main “workhorses”: the `DataArray` and the `Dataset`. Just as a Pandas `DataFrame` is composed of multiple `Series`, an Xarray `Dataset` contains one or more `DataArray` objects. We’ll start by looking at a `Dataset`, using Xarray’s representation of an ERA5 gridded data file as an example.

In [ ]:
# Load a recent ERA5 pressure level dataset
%time ds = xr.open_dataset('20260904_era5.nc')

In [ ]:
ds

#### The `ds` Dataset has the following properties:

1. It has four named dimensions: valid_time, pressure_level, latitude, and longitude.
2. It has four coordinate variables, which (in this case, but not always) correspond to the dimensions.
3. It contains 16 data variables, such as `t`, air temperature. A data variable can be read in as a Data Array.
4. Each variable has attributes which are the variable metadata and include the variable units and descriptive name.
5. The dataset together has attributes which are the data metadata
6. Its coordinate variables may also have their own metadata.

## DataArrays

In [ ]:
# Test with a dataset from 2016 that includes surface and pressure level variables for the entire geographic area (i.e., a combined Climate Data Store ERA5 file)
ds = xr.open_dataset('/spare11/atm533/data/20160123_era5.nc')
ds

When we analyze and display information in a gridded Xarray `Dataset`, we are actually working with one or more `DataArray`s within one or more `Dataset`s. In the ERA5 dataset, there are gridded fields, one of whichis `mean_sea_level_pressure`. Let's read it in as an Xarray `DataArray` object. You will see that we access it in a similar manner to how we accessed a column, aka `Series`, in a Pandas `Dataframe`.

In [ ]:
slp = ds['mean_sea_level_pressure']

In [ ]:
slp

### Similar to (but not exactly the same as) the `ds` Dataset  which contains it, this `DataArray` has the following properties:

1. It is a named *data variable*: `mean_sea_level_pressure`
2. It has three named *dimensions*, in order: time, latitude, longitude.
3. It has three *coordinate variables*, which (in this case, but not always) correspond to the *dimensions*.
4. It has *attributes* which are the data variable's *metadata*.
5. Its coordinate variables may have their own metadata as well.

### Examine each of these five properties.

## 1. Data Variable

The *data variable* is represented by the `DataArray` object itself. We can query various properties of it, with methods similar to Pandas.

In [ ]:
# Akin to column and row indices in Pandas:
slp.indexes

In [ ]:
slp.mean() # mean of entire DataArray across all coordiates

In [ ]:
slp.max() # maximum of entire DataArray across all coordinates

In [ ]:
slp.min() # minimum of entire DataArray across all coordinates

The following will return the lat, lon, and time of the min/max value in the DataArray (source: https://stackoverflow.com/questions/40179593/how-to-get-the-coordinates-of-the-maximum-in-xarray).

[Xarray.where](https://docs.xarray.dev/en/stable/generated/xarray.where.html)

In [ ]:
slp.where(slp==slp.min(), drop=True).coords

Similar to Pandas, we can use `loc` to select via dimension values, but note that the order of these values **must** correspond to the dimesnions' order. 

In [ ]:
slp.loc['2016-01-23-18:00:00', -73.5, 224.0]

We can alternatively, <b>(and preferably!)</b> use Xarray's `sel` indexing technique, where we specify the names of the dimension and the values we are selecting ... can be in any order ... and does not need to include all dimensions. In the cell below, we get the SLP values for this particular point for every time in the dataset.

In [ ]:
slp.sel(longitude = 286.25, latitude = 42.75)

<div class="admonition alert alert-danger">
    <p class="admonition-title" style="font-weight:bold">ERA5 longitude convention</p>
When we perform selection-based queries along coordinate dimensions in Xarray, we must use the same range of values as in the dataset. The ERA5, and ECMWF's gridded datasets in general, use a convention where longitudes are in <i>degrees East</i> of the Greenwich meridian ... i.e. they run from <b>0</b> to <b>360</b>. For the western hemisphere, longitudes will range from 180 to 360. So,instead of <b>-75</b> West, we would use 360 - 75 = <b>285</b>.
</div>

What if we passed in a set of coordinates that did not exactly match those in the dataset?

In [ ]:
slp.sel(longitude = 114.9, latitude = -23.2)

That failed, but fortunately Xarray has a handy `nearest` option when doing selections over coordinate dimensions!

In [ ]:
slp.sel(longitude = 114.9, latitude = -23.2, method='nearest')

## 2. Dimension names
In Xarray, dimensions can be thought of as extensions of Pandas` 2-d row/column indices (aka *axes*). We can assign names, or *labels*, to Pandas indexes; in Xarray, these *labeled axes* are a necessary (and excellent) feature.

In [ ]:
slp.dims

## 3. Coordinates

*Coordinate variables* in Xarray are 1-dimensional arrays that correspond to the *Data variable*'s dimensions.
In this case, `slp` has dimension coordinates of longitude, latitude, and time; each of these dimension coordinates consist of an array of values, plus metadata.

In [ ]:
slp.coords

We can assign an object to each coordinate dimension.

In [ ]:
lons = slp.longitude

In [ ]:
lats = slp.latitude

In [ ]:
time = slp.time

In [ ]:
time

## 4. The data variables will typically have attributes (metadata) attached to them.

In [ ]:
slp.attrs

In [ ]:
slp.units

## 5. The coordinate variables will likely (but not always) have metadata as well.

In [ ]:
lons.attrs

In [ ]:
time.attrs

## Plotting with Xarray

Similar to Pandas, Xarray includes a `plot` method into Matplotlib which we can use to get a quick view of the data. Here, we will construct a time series at a specific gridpoint.

In [ ]:
# Select a timeseries from the slp 
ts = slp.sel(longitude = 114.9, latitude = -23.2, method='nearest')
ts.plot();

### Area Plot

Create a quick plan-view data visualization at a particular time.

In [ ]:
slp.sel(time='2016-01-23-18:00:00').plot(figsize=(15,10));

In [ ]:
# Another example with temperature (observe how 'nearest' forces the selection to a datetime and level that exist in ds)
temperature = ds['temperature']
temperature.sel(time='2016-01-23-06:30:00', level=1100, method='nearest').plot(figsize=(15,10));

<div class="admonition alert alert-warning">
    <h2 class="admonition-title" style="font-weight:bold">Explore this and other gridded datasets!</h2>
    <h4>Now, try copying this notebook and make your own. Things to try in your own notebooks:</h4>
    <ol>
        <li>Read in a different variable (such as SST or any other variable of interest to you)</li>
        <li>Create a time series and plan-view visualization of that variable</li>
        <li>Read in, explore, and visualize data from the <a href="https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030/description">Global Ocean Physics Reanalysis</a>. An example data file from this reanalysis is <code>/spare11/atm533/data/mercatorglorys12v1_gl12_mean_20250821_R20250827.nc</code>.</li>
        <li><b><i>ADVANCED</i></b>: Use MetPy to convert the units of a variable (e.g. from Kelvin to Celsius)</li>
        <li><b><i>ADVANCED</i></b>: Use Matplotlib and Cartopy and create a well-labeled line- or filled- contour plot over cartographic features</li>
        <li><b><i>PRO</i></b>: Overlay multiple variables on the same map</li>
    </ol>
</div>

## References
- [Xarray Documentation](https://docs.xarray.dev/)
- [Project Pythia's Xarray Chapter](https://foundations.projectpythia.org/core/xarray/)
- [ATM350 Xarray Content](https://www.atmos.albany.edu/facstaff/ktyle/atm350/core/week11/01_Xarray_Intro.html)